In [ ]:
import os
import textwrap

webrtc_src_dir = os.path.join(os.getcwd(), "webrtc_build", "src")
wrapper_dir = os.path.join(webrtc_src_dir, "ffm_wrapper")
os.makedirs(wrapper_dir, exist_ok=True)

In [ ]:
header_content = textwrap.dedent("""\
    #pragma once
    #include <stdint.h>
    #include <stdbool.h>

    #if defined(_WIN32)
    #define FFM_EXPORT __declspec(dllexport)
    #else
    #define FFM_EXPORT
    #endif

    #ifdef __cplusplus
    extern "C" {
    #endif

    // --- Audio Processing ---
    typedef void* FFM_AudioProcessing;
    FFM_EXPORT FFM_AudioProcessing FFM_CreateAudioProcessing();
    FFM_EXPORT void FFM_FreeAudioProcessing(FFM_AudioProcessing apm);
    FFM_EXPORT void FFM_ApplyAudioProcessingConfig(FFM_AudioProcessing apm,
        bool ns_enabled, int ns_level,
        bool aec_enabled, bool aec_hp_filter,
        bool hp_enabled,
        bool gc_enabled, int gc_fixed_gain_db,
        bool agc_enabled, int agc_target_level, int agc_max_gain, int agc_max_noise, int agc_step);
    FFM_EXPORT void FFM_SetStreamDelayMs(FFM_AudioProcessing apm, int delay_ms);
    FFM_EXPORT int FFM_ProcessStream(FFM_AudioProcessing apm, const int16_t* src, int sample_rate_in, int channels, int16_t* dest);

    // --- VAD ---
    typedef void* FFM_Vad;
    FFM_EXPORT FFM_Vad FFM_CreateVad();
    FFM_EXPORT void FFM_FreeVad(FFM_Vad vad);
    FFM_EXPORT int FFM_InitVad(FFM_Vad vad);
    FFM_EXPORT int FFM_SetVadMode(FFM_Vad vad, int mode);
    FFM_EXPORT int FFM_ProcessVad(FFM_Vad vad, int sample_rate, const int16_t* audio_frame, size_t frame_length);

    // --- Resampler ---
    typedef void* FFM_Resampler;
    FFM_EXPORT FFM_Resampler FFM_CreateResampler();
    FFM_EXPORT void FFM_FreeResampler(FFM_Resampler resampler);
    FFM_EXPORT int FFM_Resample(FFM_Resampler resampler, const int16_t* src, size_t src_length, int src_rate, int16_t* dest, size_t dest_capacity, int dest_rate, int channels);

    #ifdef __cplusplus
    }
    #endif
    """)

with open(os.path.join(wrapper_dir, "webrtc_ffm.h"), "w") as f:
    f.write(header_content)

In [ ]:
cpp_content = textwrap.dedent("""\
    #include "webrtc_ffm.h"
    #include "modules/audio_processing/include/audio_processing.h"
    #include "common_audio/vad/include/webrtc_vad.h"
    #include "common_audio/resampler/include/push_resampler.h"
    #include "api/audio/audio_frame.h"

    using namespace webrtc;

    extern "C" {

    FFM_AudioProcessing FFM_CreateAudioProcessing() {
        rtc::scoped_refptr<AudioProcessing> apm = AudioProcessingBuilder().Create();
        apm->AddRef();
        return apm.get();
    }

    void FFM_FreeAudioProcessing(FFM_AudioProcessing apm) {
        if (apm) {
            static_cast<AudioProcessing*>(apm)->Release();
        }
    }

    void FFM_ApplyAudioProcessingConfig(FFM_AudioProcessing apm,
        bool ns_enabled, int ns_level,
        bool aec_enabled, bool aec_hp_filter,
        bool hp_enabled,
        bool gc_enabled, int gc_fixed_gain_db,
        bool agc_enabled, int agc_target_level, int agc_max_gain, int agc_max_noise, int agc_step) {

        AudioProcessing* ap = static_cast<AudioProcessing*>(apm);
        AudioProcessing::Config config = ap->GetConfig();

        config.noise_suppression.enabled = ns_enabled;
        config.noise_suppression.level = static_cast<AudioProcessing::Config::NoiseSuppression::Level>(ns_level);

        config.echo_canceller.enabled = aec_enabled;
        config.echo_canceller.enforce_high_pass_filtering = aec_hp_filter;

        config.high_pass_filter.enabled = hp_enabled;

        config.gain_controller1.enabled = gc_enabled || agc_enabled;
        if (gc_enabled && !agc_enabled) {
            config.gain_controller1.mode = AudioProcessing::Config::GainController1::kFixedDigital;
        } else if (agc_enabled) {
            config.gain_controller1.mode = AudioProcessing::Config::GainController1::kAdaptiveDigital;
        }

        ap->ApplyConfig(config);
    }

    void FFM_SetStreamDelayMs(FFM_AudioProcessing apm, int delay_ms) {
        static_cast<AudioProcessing*>(apm)->set_stream_delay_ms(delay_ms);
    }

    int FFM_ProcessStream(FFM_AudioProcessing apm, const int16_t* src, int sample_rate_in, int channels, int16_t* dest) {
        AudioProcessing* ap = static_cast<AudioProcessing*>(apm);
        StreamConfig config(sample_rate_in, channels);
        memcpy(dest, src, config.num_frames() * channels * sizeof(int16_t));
        return ap->ProcessStream(dest, config, config, dest);
    }

    FFM_Vad FFM_CreateVad() {
        VadInst* vad = WebRtcVad_Create();
        return vad;
    }

    void FFM_FreeVad(FFM_Vad vad) {
        if (vad) WebRtcVad_Free(static_cast<VadInst*>(vad));
    }

    int FFM_InitVad(FFM_Vad vad) {
        return WebRtcVad_Init(static_cast<VadInst*>(vad));
    }

    int FFM_SetVadMode(FFM_Vad vad, int mode) {
        return WebRtcVad_set_mode(static_cast<VadInst*>(vad), mode);
    }

    int FFM_ProcessVad(FFM_Vad vad, int sample_rate, const int16_t* audio_frame, size_t frame_length) {
        return WebRtcVad_Process(static_cast<VadInst*>(vad), sample_rate, audio_frame, frame_length);
    }

    FFM_Resampler FFM_CreateResampler() {
        return new PushResampler<int16_t>();
    }

    void FFM_FreeResampler(FFM_Resampler resampler) {
        if (resampler) delete static_cast<PushResampler<int16_t>*>(resampler);
    }

    int FFM_Resample(FFM_Resampler resampler, const int16_t* src, size_t src_length, int src_rate, int16_t* dest, size_t dest_capacity, int dest_rate, int channels) {
        PushResampler<int16_t>* r = static_cast<PushResampler<int16_t>*>(resampler);
        r->InitializeIfNeeded(src_rate, dest_rate, channels);
        return r->Resample(src, src_length, dest, dest_capacity);
    }

    }
    """)

with open(os.path.join(wrapper_dir, "webrtc_ffm.cc"), "w") as f:
    f.write(cpp_content)

In [ ]:
build_gn_content = textwrap.dedent("""\
    import("//webrtc.gni")

    shared_library("webrtc_ffm") {
      sources = [
        "webrtc_ffm.cc",
        "webrtc_ffm.h",
      ]
      deps = [
        "//modules/audio_processing:audio_processing",
        "//common_audio:common_audio",
        "//api/audio:audio_frame_api",
      ]
    }
    """)

with open(os.path.join(wrapper_dir, "BUILD.gn"), "w") as f:
    f.write(build_gn_content)

In [ ]:
root_build_gn = os.path.join(webrtc_src_dir, "BUILD.gn")
with open(root_build_gn, "a") as f:
    f.write("\n")
    f.write("group(\"ffm_build\") {\n")
    f.write("  deps = [ \"//ffm_wrapper:webrtc_ffm\" ]\n")
    f.write("}\n")